In [351]:
# reading the retail data set
retail <- read.csv("OnlineRetail.csv")

In [352]:
str(retail)

'data.frame':	541909 obs. of  8 variables:
 $ InvoiceNo  : chr  "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : int  6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: chr  "12/1/2010 8:26" "12/1/2010 8:26" "12/1/2010 8:26" "12/1/2010 8:26" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : int  17850 17850 17850 17850 17850 17850 17850 17850 17850 13047 ...
 $ Country    : chr  "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


In [353]:
df <- data.frame(retail)
head(df) 

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<int>,<chr>
1,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
2,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
3,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
4,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
5,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom
6,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850,United Kingdom


## Data Cleaning Plan

- Handle missing Customer IDs:
  - Replace with `"Unknown"`

In [354]:
any(is.na(df$CustomerID)) # I am checking this because I know there is null, I loaded this csv into local database and when selecting "CustomerID" as a PK it throwed error
df$CustomerID[is.na(df$CustomerID)] <- "Unknown"
any(is.na(df$CustomerID)) # any is used to determine if at least one element withthin a logical vector evulates to TRUE

[1] TRUE

[1] FALSE

In [355]:
any(is.na(df)) # We have no missing values in our data frame

[1] FALSE

## Abbreviation
    . We count manually replace every country but it will take time, So I looked up for library.
    . Found a library called "countrycode"
    . In our data like  RSA Republic of South Africa are there let's clearn that up, so that library does not throw warning
    . Here few data are of Islands, let's group that to different category, European Community, Channel Islands and Unspecified to other categories
    . On one side we have filtered based on known name of country "specified_country" and in other where there is islands and another stuff we added them into seprate data frame "specified_country"

In [356]:
library(countrycode)
library(dplyr) 
# for filter, %in% check element of vector a are present in vector b, it reutns a boolean vector

df$Country[df$Country == "RSA"] <- "South Africa"
df$Country[df$Country == "EIRE"] <- "Ireland"

# how do we know that, well from error using countrycode
unspecified_country <- dplyr::filter(df,  Country %in% c("European Community", "Channel Islands", "Unspecified"))

specified_country <-  dplyr::filter(df,  !Country %in% c("European Community", "Channel Islands", "Unspecified"))

# "country.name" Full English country name, destination is the format of our output, ISO 3-letter code
country_codes <- countrycode(specified_country$Country, origin = "country.name", destination = "iso3c") 

# let's update the country name in "specified_country" 
specified_country$Country <- country_codes

df <- rbind(specified_country, unspecified_country) # as the name says it's binding the rows with came dim cols

## Let's check for the quantity column
    . Here we have negative value, physically item sold cannot be negative, so we are taking the absolute value
    . I looked up for the entry to be exact zero but found none.

In [357]:
df$Quantity <- abs(df$Quantity) # assuming it a type and conversting to a absolute value 
paste("Quantity with exact zero: ",  sum(df[df$Quantity == 0]), " Quantity with non negative: ", sum(df$Quantity)) # we expect larger number than the dim of df, cause more than one items can be sold

[1] "Quantity with exact zero:  0  Quantity with non negative:  6145512"

## Checking for StockCode

In [358]:
any(is.na(df$StockCode)) # We don't have any nulls in stock code.
any(df$StockCode <= 0)

[1] FALSE

[1] FALSE

## Unit price error check
    . 2517 items have UnitPrice of zero.
    . We can find similar product and add the price.
    . If not then we need to drop the rows

In [359]:
price_not_known <- df[df$UnitPrice == 0, ]
str(price_not_known)


'data.frame':	2515 obs. of  8 variables:
 $ InvoiceNo  : chr  "536414" "536545" "536546" "536547" ...
 $ StockCode  : chr  "22139" "21134" "22145" "37509" ...
 $ Description: chr  "" "" "" "" ...
 $ Quantity   : int  56 1 1 1 1 1 1 3 23 10 ...
 $ InvoiceDate: chr  "12/1/2010 11:52" "12/1/2010 14:32" "12/1/2010 14:33" "12/1/2010 14:33" ...
 $ UnitPrice  : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CustomerID : chr  "Unknown" "Unknown" "Unknown" "Unknown" ...
 $ Country    : chr  "GBR" "GBR" "GBR" "GBR" ...
